In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot   as plt
# for historical data
import seaborn as sns

In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
df

## EDA (Exploratory Data Analysis)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
numerical = ['age', 'bmi', 'children', 'charges']

In [ ]:
for col in numerical:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col],kde=True,bins=20)

In [ ]:
sns.countplot(x= df['children'])

In [ ]:
sns.countplot(x= df['sex'])

In [ ]:
for col in numerical:
    plt.figure(figsize=(8,6))
    sns.boxplot(x=df[col])

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True)


### Data preprocessing and cleaning

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.drop_duplicates(inplace=True)

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({'male':0,'female':1})

In [ ]:
df_cleaned.head(5)

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes':1,'no':0})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    'sex':'is_female',
    'smoker':'is_smoker'
},inplace=True)

In [ ]:
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned,columns=['region'])

In [ ]:
df_cleaned

In [ ]:
df_cleaned.astype(int)

### Feature engineering and extraction

In [ ]:
sns.histplot(df_cleaned['bmi'])

In [ ]:
df_cleaned['catagory_bmi'] = pd.cut(
    df_cleaned['bmi'],
    bins=[0,18.5,24.9,29.9,float('inf')],
    labels=['underWeight','Normal','OverWeight','Obese']

)

In [ ]:
df_cleaned

In [ ]:
df_cleaned= pd.get_dummies(df_cleaned, columns=['catagory_bmi'])

In [ ]:
df_cleaned.astype(int)

In [ ]:
# standard devation of all (distance from the mean )

from sklearn.preprocessing import StandardScaler
cols=['age','bmi','children']

scalar = StandardScaler()



In [ ]:
df_cleaned[cols] = scalar.fit_transform(df_cleaned[cols])

In [ ]:
df_cleaned

In [ ]:
from scipy.stats import  pearsonr

# Pearson Correlation Calculation

selected_features = ['age','is_female','bmi','children','is_smoker','region_northeast','region_northwest','region_southeast','region_southwest','catagory_bmi_underWeight','catagory_bmi_Normal','catagory_bmi_OverWeight','catagory_bmi_Obese']

correlation = {
    feature:pearsonr(df_cleaned[feature],df_cleaned['charges'])[0] for feature in selected_features
}

In [ ]:
correlation.items()

In [ ]:
correlation_df=pd.DataFrame(list(correlation.items()),columns=['Feature', 'Pearson Correlation'])


In [ ]:
correlation_df.sort_values(by='Pearson Correlation' ,ascending=False)

In [ ]:
df_cleaned

In [ ]:
cat_features = [
    'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest','region_northeast',
    'catagory_bmi_Normal', 'catagory_bmi_OverWeight', 'catagory_bmi_Obese','catagory_bmi_underWeight'
]

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }
    
# converting into data fame 
chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by='p_value')
chi2_df

In [ ]:
final_df = df_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker', 'charges','region_southeast','catagory_bmi_Obese']]

In [ ]:
final = final_df.copy()

In [ ]:
from sklearn.model_selection import train_test_split
final['region_southeast'] = final['region_southeast'].astype(int)
final['catagory_bmi_Obese'] = final['catagory_bmi_Obese'].astype(int)
final

In [ ]:
x = final.drop('charges',axis=1)
y = final['charges']
x

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x,  y, test_size=.20,random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(x_train,y_train)

In [ ]:
x = np.array([[35, 1, 28.5, 2, 0, 1, 0]])


In [ ]:
y_pred = model.predict(x_test)

In [ ]:
y_pred

In [ ]:
y_pred[0]

In [ ]:
y_test

In [299]:
from sklearn.metrics import r2_score

#### test by r^2 score

In [ ]:
r^2= r2_score(y_test,y_pred)

0.8040413900197302

In [312]:
# Adjusted R^2 test 
n = x_test.shape[0]
p = x_test.shape[1]

In [ ]:
adjusted_r2 = 1-()

,age,is_female,bmi,children,is_smoker,region_southeast,catagory_bmi_Obese
900,0.696474,0,-1.336209,-0.909234,0,0,0
1064,-0.728120,1,-0.830321,2.409936,0,0,0
1256,0.838934,1,0.938238,1.580143,0,0,1
298,-0.585661,0,0.611091,1.580143,1,0,1
237,-0.585661,0,1.267024,0.750351,0,1,1
...,...,...,...,...,...,...,...
534,1.764921,0,1.609749,-0.909234,0,1,1
542,1.693691,1,0.924299,-0.909234,0,1,1
760,-1.226729,1,0.642248,0.750351,0,0,1
1284,1.551231,0,0.924299,-0.079442,1,0,1
